# Multi-Chain Gibbs Sampler - Complete Analysis

This notebook demonstrates how to use the complete framework for multi-chain MCMC sampling with:
- **Any reduced dimension D (D=1, 2, 3, 5, ...)**
- **Any layer architecture (1-layer, 2-layer, 3-layer)**
- **Preset configurations for common D×Layer combinations**
- **Full control over all hyperparameters**
- **Comprehensive diagnostics and visualizations**

## 📋 How to Run This Notebook

### Step 1: Setup and Imports
Run the first cell to import all necessary modules. Make sure you're in the `github_results` directory.

### Step 2: Choose Your Configuration Method
You have three options:

1. **Use preset config functions** (easiest) - Recommended for beginners
   - Example: `create_config_D1_L1(p=10, seed=42, n_iterations=2000)`
   - Automatically sets sensible defaults
   - Supports automatic initialization when `p` is provided

2. **Use `get_config_for(D, layer, **overrides)`** (flexible)
   - Example: `get_config_for(D=2, layer=2, n_iterations=1000, use_mle_all=True)`
   - Good for quick experiments with custom settings

3. **Specify all parameters manually** (full control)
   - Pass all parameters directly to `run_multichain_analysis()`
   - Use when you need complete control over every hyperparameter

### Step 3: Prepare Your Data
- **Training data**: `Y_train` (n,), `X_train` (n, p)
- **Test data**: `Y_test` (n_test,), `X_test` (n_test, p)
- Or use the provided data generation functions

### Step 4: Run Analysis
Call `run_multichain_analysis()` with your configuration and data.

### Step 5: Analyze Results
- Check convergence diagnostics (R-hat < 1.1 indicates convergence)
- Review performance metrics (RMSPE, NSME, CRPS, BIC, MLPPD)
- Examine diagnostic plots in the output directory

## 🚀 Quick Start Examples

### Example 1: Simple 1-Layer GP (D=1)
```python
# Generate data
data = generate_case1_1d(n=200, seed=42)

# Use preset config
config = create_config_D1_L1(p=data['X_train'].shape[1], seed=42)

# Run analysis
results = run_multichain_analysis(
    Y_train=data['y_train'],
    X_train=data['X_train'],
    Y_test=data['y_test'],
    X_test=data['X_test'],
    **config
)
```

### Example 2: 2-Layer Deep GP (D=2)
```python
# Generate data
data = generate_case1_2d(n=200, seed=42)

# Use preset config with automatic initialization
config = create_config_D2_L2(
    p=data['X_train'].shape[1],
    seed=42,
    n_iterations=2000,
    use_mle_all=True
)

# Run analysis
results = run_multichain_analysis(
    Y_train=data['y_train'],
    X_train=data['X_train'],
    Y_test=data['y_test'],
    X_test=data['X_test'],
    **config
)
```

### Example 3: Using Variants (Simplified Models)
```python
# Layer 1 variant: W is known
W_fixed = np.random.randn(p, 2)
W_fixed, _ = np.linalg.qr(W_fixed)

config = create_config_L1_W_Known(W_fixed=W_fixed, n_iterations=1000)

results = run_multichain_analysis(
    Y_train=Y_train, X_train=X_train,
    Y_test=Y_test, X_test=X_test,
    layer=1, variant='W_Known',
    **config
)
```

## Important Parameters

### Model Specification
- **D**: Reduced dimension (1, 2, 3, 5, ...)
- **layer**: Number of layers (1, 2, or 3)
- **variant**: Optional - 'W_Known', 'No_W', or 'No_W_Selective' (for simplified models)

### MCMC Settings
- **n_chains**: Number of independent chains (default: 3, recommended: 3-5)
- **n_iterations**: Total iterations per chain (default: 2000, recommended: 2000-10000)
- **burn_in**: Burn-in period to discard (default: 500, recommended: 500-2000)
- **thin**: Thinning interval (default: 2, recommended: 1-5)

### Estimation Options
- **use_mle_all**: Use MLE for all hyperparameters (fastest, default: False)
- **use_mle_tau2**: Use MLE for tau2 only (Layer 1)
- **use_mle_g**: Use MLE for g only (Layer 1)
- **use_mle_theta**: Use MLE for theta only (Layer 1)
- **use_mle_tau2**, **use_mle_g_y**, **use_mle_theta_y**: For Layer 2/3 (Y layer only)

### Output Options
- **output_dir**: Directory to save plots and diagnostics (default: './diagnostics')
- **save_plots**: Save diagnostic plots (default: True)
- **save_samples**: Save MCMC samples (default: True)
- **verbose**: Print progress (default: True)

## 📊 Understanding Results

### Convergence Diagnostics
- **R-hat (Gelman-Rubin)**: Should be < 1.1 for convergence
- **Heidelberg-Welch**: Tests for stationarity

### Performance Metrics
- **RMSPE**: Lower is better (predictive accuracy)
- **NSME**: Higher is better, max = 1 (model efficiency)
- **CRPS**: Lower is better (probabilistic accuracy)
- **BIC**: higher is better (model selection)
- **MLPPD**: Higher is better (predictive quality)

### Diagnostic Plots
Check the `output_dir` for:
- Trace plots (check for convergence)
- Density plots (posterior distributions)
- Autocorrelation plots (check for mixing)
- Actual vs Predicted plots (model fit)

## 💡 Tips for Best Results

1. **Start with MLE mode** (`use_mle_all=True`) for quick exploration
2. **Use automatic initialization** by providing `p` parameter to preset configs
3. **Check convergence** before interpreting results (R-hat < 1.1)
4. **Use TensorFlow gradients** for D>1 (`use_tf_gradients=True`)
5. **Increase iterations** if convergence is not reached
6. **Use variants** when W/M/Lambda/V are not of interest (faster sampling)


## 📦 Step 1: Import Required Modules

The following cell imports all necessary modules and functions. Make sure you're running this notebook from the `github_results` directory.


In [ ]:
import numpy as np
import sys
from pathlib import Path
from scipy.linalg import svd
from typing import Optional

# Add module paths
base_dir = Path.cwd()
for folder in ["Multichain", "Gibbs Sampling", "Parameter Sampler", "BDR Metrics and Plot", "Data Generation"]:
    sys.path.insert(0, str(base_dir / folder))

# Import main function and config helpers
from run_multichains import (
    run_multichain_analysis,
    get_config_for,
    create_config_D1_L1, create_config_D1_L2, create_config_D1_L3,
    create_config_D2_L1, create_config_D2_L2, create_config_D2_L3,
    create_config_D3_L1, create_config_D3_L2, create_config_D3_L3,
    create_config_D5_L1, create_config_D5_L2, create_config_D5_L3,
    create_config_L1_W_Known, create_config_L1_No_W, create_config_L1_No_W_Selective,
    create_config_L2_W_Known, create_config_L2_No_W, create_config_L2_No_W_Selective,
    create_config_L3_W_Known, create_config_L3_No_W, create_config_L3_No_W_Selective,
    initialize_M_Lambda_V_W_D1,
    initialize_M_Lambda_V_W_Dgeneral
)
from Data_generation import generate_case1_1d, generate_case1_2d
from parameter_sampler_D1 import rmf_matrixN, rmf_matrix
from parameter_sampler_Dgeneral import rmf_matrixN as rmf_matrixN_Dgeneral, rmf_matrix as rmf_matrix_Dgeneral

print("✅ Imports successful!")
print("\n📋 Available preset configs:")
print("   D=1: create_config_D1_L1/L2/L3(p, seed, **kwargs)")
print("   D=2: create_config_D2_L1/L2/L3(p, seed, **kwargs)")
print("   D=3: create_config_D3_L1/L2/L3(p, seed, **kwargs)")
print("   D=5: create_config_D5_L1/L2/L3(p, seed, **kwargs)")
print("\n   Layer 1 Variants:")
print("   - create_config_L1_W_Known(W_fixed, **kwargs)")
print("   - create_config_L1_No_W(**kwargs)")
print("   - create_config_L1_No_W_Selective(D, column_indices=None, **kwargs)")
print("\n   Or use: get_config_for(D, layer, **overrides)")
print("\n💡 New: All config functions accept 'p' parameter for automatic initialization!")
print("   Example: config = create_config_D2_L1(p=10, seed=42)")


## 🔧 Step 2: Understanding Initialization Functions

The framework provides automatic initialization functions that set up M, Lambda, V, W, and their priors using an SVD-based procedure. This ensures consistent starting values when the input dimension `p` is known.

**When to use:**
- Provide `p` parameter to preset config functions (e.g., `create_config_D2_L1(p=10, seed=42)`)
- The initialization happens automatically inside the config function
- For manual initialization, use `initialize_M_Lambda_V_W_D1()` or `initialize_M_Lambda_V_W_Dgeneral()`


In [ ]:
# Demonstrate the initialization functions

print("="*70)
print("Demonstrating Initialization Functions")
print("="*70)

# Example 1: D=1 initialization
p = 10
print(f"\nExample 1: D=1, p={p}")
init_D1 = initialize_M_Lambda_V_W_D1(p=p, D=1, seed=42)
print(f"  M_init shape: {init_D1['M_init'].shape}")
print(f"  W_init shape: {init_D1['W_init'].shape}")
print(f"  prior_M shape: {init_D1['prior_M'].shape}")
print(f"  prior_V shape: {init_D1['prior_V'].shape}")
print(f"  Lambda_init: {init_D1['Lambda_init']}")

# Example 2: D=2 initialization
print(f"\nExample 2: D=2, p={p}")
init_D2 = initialize_M_Lambda_V_W_Dgeneral(p=p, D=2, seed=42)
print(f"  M_init shape: {init_D2['M_init'].shape}")
print(f"  W_init shape: {init_D2['W_init'].shape}")
print(f"  prior_M shape: {init_D2['prior_M'].shape}")
print(f"  prior_V shape: {init_D2['prior_V'].shape}")
print(f"  Lambda_init shape: {init_D2['Lambda_init'].shape}")
print(f"  prior_Lambda:\n{init_D2['prior_Lambda']}")

# Example 3: D=3 initialization
print(f"\nExample 3: D=3, p={p}")
init_D3 = initialize_M_Lambda_V_W_Dgeneral(p=p, D=3, seed=42)
print(f"  W_init shape: {init_D3['W_init'].shape}")
print(f"  prior_Lambda:\n{init_D3['prior_Lambda']}")

print(f"\n✅ All initialization functions work correctly!")
print("="*70)


## 📊 Step 3: Running Your First Analysis

### Method 1: Using Preset Configurations (Recommended)

Preset configurations are the easiest way to get started. They provide sensible defaults and support automatic initialization.

**Steps:**
1. Generate or load your data
2. Choose a preset config function (e.g., `create_config_D1_L1`)
3. Optionally provide `p` for automatic initialization
4. Override any parameters you want to change
5. Pass the config to `run_multichain_analysis()`


## Initialization Functions

The framework now includes automatic initialization of M, Lambda, V, and W using SVD-based procedures:

### For D=1:
- `initialize_M_Lambda_V_W_D1(p, D=1, seed=None)`
- Uses: `F = np.random.randn(p, D)`, SVD, then computes priors and W_init

### For D>1 (D=2, 3, 5, ...):
- `initialize_M_Lambda_V_W_Dgeneral(p, D, seed=None)`
- Uses: `F = np.random.randn(p, D)`, SVD, then computes priors and W_init
- `prior_Lambda` is drawn diagonally from `Gamma(5/2, 10/3)`

### Configuration Functions

All configuration functions now accept `p` and `seed` parameters:
- If `p` is provided, M, Lambda, V, W, and priors are automatically initialized
- If `p` is None, defaults are used (random initialization in samplers)


## 📊 Step 4: Running Analysis with D>1

For D>1 cases, the framework automatically handles:
- Vector theta parameters (one per dimension)
- Separable kernels (dimension-wise lengthscales)
- TensorFlow gradients (recommended for better performance)

**Key differences from D=1:**
- `theta_y_init` should be a vector of shape `(D,)`
- `use_tf_gradients=True` is recommended
- Kernel type defaults to `'separable_squared_exponential'` for D>1


## Method 1: Using Preset Configuration Functions with Automatic Initialization

The easiest way! Use `create_config_D{D}_L{layer}(p, seed, **kwargs)` functions.

**New Feature**: All config functions now accept `p` (input dimension) parameter:
- If `p` is provided: M, Lambda, V, W, and priors are automatically initialized using SVD procedure
- If `p` is None: Defaults are used (random initialization in samplers)

### Initialization Procedure:

**For D=1:**
1. `F = np.random.randn(p, D)`
2. `Mm_init, Ll, V_init = svd(F)`
3. `M_init = Mm_init[:, :D]`
4. `Lambda_init = np.diag(Ll)`
5. `prior_M = rmf_matrixN(M=M_init.reshape(-1, 1))`
6. `prior_V = rmf_matrix(M=V_init)`
7. `W_init = rmf_matrixN(M=(M_init @ Lambda_init) @ V_init)`

**For D>1 (D=2, 3, 5, ...):**
1. `F = np.random.randn(p, D)`
2. `Mm_init, Ll, V_init = svd(F)`
3. `M_init = Mm_init[:, :D]`
4. `Lambda_init = np.diag(Ll)`
5. `prior_M = rmf_matrixN(M=M_init)`
6. `prior_V = rmf_matrix(M=V_init)`
7. `prior_Lambda` is drawn diagonally from `Gamma(5/2, 10/3)`
8. `W_init = rmf_matrixN(M=(M_init @ Lambda_init) @ V_init.T)`


## 📋 Step 5: Available Preset Configurations

The framework provides preset configurations for all common D × Layer combinations:

- **D=1**: `create_config_D1_L1`, `create_config_D1_L2`, `create_config_D1_L3`
- **D=2**: `create_config_D2_L1`, `create_config_D2_L2`, `create_config_D2_L3`
- **D=3**: `create_config_D3_L1`, `create_config_D3_L2`, `create_config_D3_L3`
- **D=5**: `create_config_D5_L1`, `create_config_D5_L2`, `create_config_D5_L3`

**All preset functions:**
- Accept `p` parameter for automatic initialization
- Accept `seed` parameter for reproducibility
- Accept `**kwargs` to override any default parameters


In [ ]:
# Generate data for D=1
data_D1 = generate_case1_1d(n=200, seed=42)

# Get input dimension
p = data_D1['X_train'].shape[1]

# Get preset config for D=1, Layer=1 with automatic initialization
# By providing 'p', M, Lambda, V, W, and priors are automatically initialized
config = create_config_D1_L1(
    p=p,  # Provide p for automatic initialization
    seed=42,  # Random seed for reproducibility
    n_iterations=1000,
    burn_in=200,
    use_mle_all=False,  # Fast mode
    output_dir='./results_D1_L1'
)

print("Configuration for D=1, Layer=1:")
print(f"  D={config['D']}, Layer={config['layer']}")
print(f"  Iterations={config['n_iterations']}, Burn-in={config['burn_in']}")
print(f"  MLE mode={config['use_mle_all']}")
print(f"  theta_y_init={config['theta_y_init']} (scalar for D=1)")
print(f"\n  Automatic Initialization (p={p} provided):")
print(f"    W_init is not None: {config['W_init'] is not None}")
print(f"    prior_M is not None: {config['prior_M'] is not None}")
print(f"    prior_V is not None: {config['prior_V'] is not None}")
if config['W_init'] is not None:
    print(f"    W_init shape: {config['W_init'].shape}")
    print(f"    W_init norm: {np.linalg.norm(config['W_init']):.6f} (should be ~1.0)")

# Run analysis
results_D1_L1 = run_multichain_analysis(
    Y_train=data_D1['y_train'],
    X_train=data_D1['X_train'],
    Y_test=data_D1['y_test'],
    X_test=data_D1['X_test'],
    **config  # Unpack configuration
)


## 🔬 Step 6: Advanced Example - 3-Layer Deep GP

3-layer models are the most complex, with three levels of latent variables:
- **Y layer** (outer): tau2_y, g_y, theta_y
- **Q layer** (middle): Q (ESS), theta_q
- **R layer** (inner): R (ESS), theta_r

**Hierarchical lengthscale priors:**
- `gamma2_y = 3.9` (outer layer)
- `gamma2_q = 3.9/3` (middle layer)
- `gamma2_r = 3.9/6` (inner layer)

**Note:** For 3-layer models, sampling takes longer due to multiple ESS steps per iteration.


## Method 2: Using `get_config_for()` Helper

Convenient wrapper for any D and layer combination. Note: This doesn't support automatic initialization yet - use preset functions with `p` parameter for that.


## 📈 Step 7: Analyzing Results

After running `run_multichain_analysis()`, you get a results dictionary with:

1. **`chains_samples`**: List of dictionaries, one per chain, containing all MCMC samples
2. **`chains_metrics`**: List of dictionaries, one per chain, containing performance metrics
3. **`convergence`**: Dictionary with R-hat statistics and convergence diagnostics
4. **`metrics_summary`**: Dictionary with summary statistics (mean, median, std, CI) for all metrics
5. **`computation_times`**: List of computation times per chain

**Key things to check:**
- R-hat values should be < 1.1 for convergence
- Check trace plots for good mixing
- Review performance metrics to assess model quality


In [ ]:
# Generate data for D=2
data_D2 = generate_case1_2d(n=200, seed=42)

# Get input dimension
p = data_D2['X_train'].shape[1]

# Use preset config for D=2, Layer=2 with automatic initialization
config_D2_L2 = create_config_D2_L2(
    p=p,  # Provide p for automatic initialization
    seed=42,
    n_iterations=1000,
    use_mle_all=True,
    use_tf_gradients=True,  # Recommended for D>1
    output_dir='./results_D2_L2'
)

print("Configuration for D=2, Layer=2:")
print(f"  D={config_D2_L2['D']}, Layer={config_D2_L2['layer']}")
print(f"  theta_y_init shape: {config_D2_L2['theta_y_init'].shape} (vector for D>1)")
print(f"  theta_q_init shape: {config_D2_L2['theta_q_init'].shape}")
print(f"  TensorFlow gradients: {config_D2_L2['use_tf_gradients']}")
print(f"\n  Automatic Initialization (p={p} provided):")
print(f"    W_init is not None: {config_D2_L2['W_init'] is not None}")
print(f"    prior_M is not None: {config_D2_L2['prior_M'] is not None}")
print(f"    prior_V is not None: {config_D2_L2['prior_V'] is not None}")
if config_D2_L2['W_init'] is not None:
    print(f"    W_init shape: {config_D2_L2['W_init'].shape}")
    print(f"    W_init column norms: {np.linalg.norm(config_D2_L2['W_init'], axis=0)}")

results_D2_L2 = run_multichain_analysis(
    Y_train=data_D2['y_train'],
    X_train=data_D2['X_train'],
    Y_test=data_D2['y_test'],
    X_test=data_D2['X_test'],
    **config_D2_L2
)


## All Available Preset Configurations

Quick reference for all D × Layer combinations with presets.


In [ ]:
import pandas as pd

# Show all available configurations
configs_info = []

for D in [1, 2, 3, 5]:
    for layer in [1, 2, 3]:
        config = get_config_for(D, layer)
        configs_info.append({
            'D': D,
            'Layer': layer,
            'Function': f'create_config_D{D}_L{layer}(p, seed, **kwargs)',
            'TF Gradients': config['use_tf_gradients'],
            'MLE Default': config['use_mle_all'],
            'theta shape': 'scalar' if D == 1 else f'({D},)',
            'Auto Init': 'Yes (if p provided)'
        })

df = pd.DataFrame(configs_info)
print("Available Preset Configurations:")
print("="*100)
print(df.to_string(index=False))
print("\nUsage with automatic initialization:")
print("   config = create_config_D2_L1(p=10, seed=42, n_iterations=5000, use_mle_all=True)")
print("\nUsage without automatic initialization:")
print("   config = create_config_D2_L1(n_iterations=5000, use_mle_all=True)")
print("   Or: config = get_config_for(D=2, layer=1, **overrides)")


## Example: D=3, 3-Layer Deep GP

Demonstrate higher dimensional case with deep architecture and automatic initialization.


In [ ]:
# Generate D=3 data
np.random.seed(42)
n, p, D = 200, 5, 3
X_train_D3 = np.random.randn(n, p)
X_test_D3 = np.random.randn(50, p)
W_true = np.random.randn(p, D)
W_true, _ = np.linalg.qr(W_true)
Y_train_D3 = np.sin((X_train_D3 @ W_true).sum(axis=1)) + 0.1 * np.random.randn(n)
Y_test_D3 = np.sin((X_test_D3 @ W_true).sum(axis=1)) + 0.1 * np.random.randn(50)

# Use preset for D=3, Layer=3 with automatic initialization
config_D3_L3 = create_config_D3_L3(
    p=p,  # Provide p for automatic initialization
    seed=42,
    n_iterations=1000,
    burn_in=200,
    use_mle_all=True,
    output_dir='./results_D3_L3'
)

print("Configuration for D=3, Layer=3 (3-layer Deep GP):")
print(f"  Parameters: tau2_y, g_y, g_q, g_r, theta_y, theta_q, theta_r, R, Q, W")
print(f"  theta_y_init: {config_D3_L3['theta_y_init']}")
print(f"  theta_q_init: {config_D3_L3['theta_q_init']}")
print(f"  theta_r_init: {config_D3_L3['theta_r_init']}")
print(f"  All theta vectors have shape (3,) for D=3")
print(f"\n  Automatic Initialization (p={p} provided):")
print(f"    W_init is not None: {config_D3_L3['W_init'] is not None}")
print(f"    prior_M is not None: {config_D3_L3['prior_M'] is not None}")
print(f"    prior_V is not None: {config_D3_L3['prior_V'] is not None}")
if config_D3_L3['W_init'] is not None:
    print(f"    W_init shape: {config_D3_L3['W_init'].shape}")
    print(f"    W_init column norms: {np.linalg.norm(config_D3_L3['W_init'], axis=0)}")

results_D3_L3 = run_multichain_analysis(
    Y_train=Y_train_D3,
    X_train=X_train_D3,
    Y_test=Y_test_D3,
    X_test=X_test_D3,
    **config_D3_L3
)


## Results Analysis

Access convergence diagnostics and performance metrics.


In [ ]:
# Example: Analyze results from D=1, Layer=1
print("="*70)
print("CONVERGENCE DIAGNOSTICS (D=1, Layer=1)")
print("="*70)

for key, val in results_D1_L1['convergence'].items():
    if 'r_hat' in key:
        status = "✓" if val < 1.1 else "⚠"
        print(f"{key:<20} {val:8.4f}  {status}")

print("\n" + "="*70)
print("PERFORMANCE METRICS (D=1, Layer=1)")
print("="*70)

for name, vals in results_D1_L1['metrics_summary'].items():
    print(f"\n{name.upper()}:")
    print(f"  Mean:   {vals['mean']:.4f}")
    print(f"  Median: {vals['median']:.4f}")
    print(f"  95% CI: [{vals['ci_lower']:.4f}, {vals['ci_upper']:.4f}]")

print("\n" + "="*70)
print("COMPUTATION TIME")
print("="*70)
total_time = sum(results_D1_L1['computation_times'])
print(f"Total: {total_time:.2f}s")
print(f"Per chain: {np.mean(results_D1_L1['computation_times']):.2f}s")


## Layer 1 Variants (NEW!)

The framework now supports three Layer 1 variants for simplified models that skip W, M, Lambda, V sampling:

### 1. **W_Known**: W is fixed/known
   - **Use when**: You have a known projection matrix W (e.g., from PCA or previous analysis)
   - **What's NOT sampled**: W, M, Lambda, V (all fixed/known)
   - **What IS sampled**: 
     - `tau2_y`: Observation noise variance (MLE or MCMC)
     - `g_y`: Nugget parameter (MLE or MCMC)
     - `theta_D_y`: Lengthscale parameter(s) (MLE or MCMC)
   - **Config**: `create_config_L1_W_Known(W_fixed, **kwargs)`
   - **Usage**: `run_multichain_analysis(..., layer=1, variant='W_Known', W_fixed=W_fixed, ...)`
   - **Model**: Y | X, W_fixed, θ_D_y, g_y, τ²_y ~ GP(0, τ²(C_y + g*I)) where Z = XW_fixed

### 2. **No_W**: No dimensionality reduction
   - **Use when**: You don't want dimensionality reduction, use X directly as input
   - **What's NOT sampled**: W, M, Lambda, V (not needed)
   - **What IS sampled**: 
     - `tau2_y`: Observation noise variance (MLE or MCMC)
     - `g_y`: Nugget parameter (MLE or MCMC)
     - `theta_D_y`: Lengthscale parameter(s) (MLE or MCMC)
   - **Config**: `create_config_L1_No_W(**kwargs)`
   - **Usage**: `run_multichain_analysis(..., layer=1, variant='No_W', ...)`
   - **Model**: Y | X, θ_D_y, g_y, τ²_y ~ GP(0, τ²(C_y + g*I)) where input is X directly

### 3. **No_W_Selective**: Use selected columns of X
   - **Use when**: You want to use only some columns of X (feature selection)
   - **What's NOT sampled**: W, M, Lambda, V (not needed)
   - **What IS sampled**: 
     - `tau2_y`: Observation noise variance (MLE or MCMC)
     - `g_y`: Nugget parameter (MLE or MCMC)
     - `theta_D_y`: Lengthscale parameter(s) (MLE or MCMC)
   - **Config**: `create_config_L1_No_W_Selective(D, column_indices=None, **kwargs)`
   - **Usage**: `run_multichain_analysis(..., layer=1, variant='No_W_Selective', D=3, column_indices=[0,1,2], ...)`
   - **Model**: Y | X[:, column_indices], θ, g, τ² ~ GP(0, τ²(C_y + g*I))

### All Layer 1 Variants Support:
- **Kernel type selection**: `'isotropic_squared_exponential'`, `'separable_squared_exponential'`, `'isotropic_matern32'`, `'separable_matern32'`
- **Individual MLE options**: `use_mle_tau2`, `use_mle_g`, `use_mle_theta` (independent flags, default: False = MCMC)
- **Both D=1 and D>1 cases**: Automatically handles scalar vs vector theta
- **Hyperparameters**: 
  - `alpha1`, `alpha2`: Inverse Gamma prior for tau2 (default: 1.0, 1000.0)
  - `beta1`, `beta2`: Gamma prior for g (default: 0.01, 0.005)
  - `gamma1`, `gamma2`: Gamma prior for theta (default: 1.5, 3.9)
- **Multi-chain diagnostics and metrics**: Full convergence diagnostics and performance metrics

## Layer 2 Variants (NEW!)

The framework now supports three Layer 2 variants for simplified 2-layer models that skip W, M, Lambda, V sampling:

### 1. **W_Known**: W is fixed/known
   - **Use when**: You have a known projection matrix W
   - **What's NOT sampled**: W, M, Lambda, V (all fixed/known)
   - **What IS sampled**: 
     - **Y layer**: `tau2_y` (MLE or MCMC), `g_y` (MLE or MCMC), `theta_y` (MLE or MCMC)
     - **Q layer (latent)**: `Q` (ESS - Elliptical Slice Sampling), `theta_q` (MCMC only)
     - **Q layer fixed**: `g_q = 0.0` (no nugget), `tau2_q = 1.0` (fixed variance)
   - **Config**: `create_config_L2_W_Known(W_fixed, **kwargs)`
   - **Usage**: `run_multichain_analysis(..., layer=2, variant='W_Known', W_fixed=W_fixed, ...)`
   - **Model**: Y | Q, θ_y, g_y, τ² ~ GP(0, τ²(C_y + g_y*I)); Q | Z, θ_q ~ GP(0, C_q) where Z = XW_fixed

### 2. **No_W**: No dimensionality reduction
   - **Use when**: You don't want dimensionality reduction, use X directly
   - **What's NOT sampled**: W, M, Lambda, V (not needed)
   - **What IS sampled**: 
     - **Y layer**: `tau2_y` (MLE or MCMC), `g_y` (MLE or MCMC), `theta_y` (MLE or MCMC)
     - **Q layer (latent)**: `Q` (ESS), `theta_q` (MCMC only)
     - **Q layer fixed**: `g_q = 0.0`, `tau2_q = 1.0`
   - **Config**: `create_config_L2_No_W(**kwargs)`
   - **Usage**: `run_multichain_analysis(..., layer=2, variant='No_W', ...)`
   - **Model**: Y | Q, θ_y, g_y, τ² ~ GP(0, τ²(C_y + g_y*I)); Q | X, θ_q ~ GP(0, C_q)

### 3. **No_W_Selective**: Use selected columns of X
   - **Use when**: You want to use only some columns of X
   - **What's NOT sampled**: W, M, Lambda, V (not needed)
   - **What IS sampled**: 
     - **Y layer**: `tau2_y` (MLE or MCMC), `g_y` (MLE or MCMC), `theta_y` (MLE or MCMC)
     - **Q layer (latent)**: `Q` (ESS), `theta_q` (MCMC only)
     - **Q layer fixed**: `g_q = 0.0`, `tau2_q = 1.0`
   - **Config**: `create_config_L2_No_W_Selective(D, column_indices=None, **kwargs)`
   - **Usage**: `run_multichain_analysis(..., layer=2, variant='No_W_Selective', D=3, column_indices=[0,1,2], ...)`
   - **Model**: Y | Q, θ_y, g_y, τ² ~ GP(0, τ²(C_y + g_y*I)); Q | X[:, column_indices], θ_q ~ GP(0, C_q)

### All Layer 2 Variants Support:
- **Kernel type selection**: All four kernel types supported
- **Individual MLE options**: Only for Y layer (`use_mle_tau2`, `use_mle_g_y`, `use_mle_theta_y`)
- **Q layer**: Always uses MCMC for `theta_q` (no MLE option)
- **Hierarchical lengthscale priors**: 
  - `gamma2_y = 3.9` (outer layer Y)
  - `gamma2_q = 3.9/3` (middle layer Q)
- **Both D=1 and D>1 cases**: Handles scalar vs vector theta automatically
- **Multi-chain diagnostics and metrics**: Full support

## Layer 3 Variants (NEW!)

The framework now supports three Layer 3 variants for simplified 3-layer models that skip W, M, Lambda, V sampling:

### 1. **W_Known**: W is fixed/known
   - **Use when**: You have a known projection matrix W
   - **What's NOT sampled**: W, M, Lambda, V (all fixed/known)
   - **What IS sampled**: 
     - **Y layer**: `tau2_y` (MLE or MCMC), `g_y` (MLE or MCMC), `theta_y` (MLE or MCMC)
     - **Q layer (middle latent)**: `Q` (ESS), `theta_q` (MCMC only)
     - **Q layer fixed**: `g_q = 0.0`, `tau2_q = 1.0`
     - **R layer (inner latent)**: `R` (ESS), `theta_r` (MCMC only)
     - **R layer fixed**: `g_r = 0.0`, `tau2_r = 1.0`
   - **Config**: `create_config_L3_W_Known(W_fixed, **kwargs)`
   - **Usage**: `run_multichain_analysis(..., layer=3, variant='W_Known', W_fixed=W_fixed, ...)`
   - **Model**: Y | Q, θ_y, g_y, τ² ~ GP(0, τ²(C_y + g_y*I)); Q | R, θ_q ~ GP(0, C_q); R | Z, θ_r ~ GP(0, C_r) where Z = XW_fixed

### 2. **No_W**: No dimensionality reduction
   - **Use when**: You don't want dimensionality reduction, use X directly
   - **What's NOT sampled**: W, M, Lambda, V (not needed)
   - **What IS sampled**: 
     - **Y layer**: `tau2_y` (MLE or MCMC), `g_y` (MLE or MCMC), `theta_y` (MLE or MCMC)
     - **Q layer (middle latent)**: `Q` (ESS), `theta_q` (MCMC only)
     - **Q layer fixed**: `g_q = 0.0`, `tau2_q = 1.0`
     - **R layer (inner latent)**: `R` (ESS), `theta_r` (MCMC only)
     - **R layer fixed**: `g_r = 0.0`, `tau2_r = 1.0`
   - **Config**: `create_config_L3_No_W(**kwargs)`
   - **Usage**: `run_multichain_analysis(..., layer=3, variant='No_W', ...)`
   - **Model**: Y | Q, θ_y, g_y, τ² ~ GP(0, τ²(C_y + g_y*I)); Q | R, θ_q ~ GP(0, C_q); R | X, θ_r ~ GP(0, C_r)

### 3. **No_W_Selective**: Use selected columns of X
   - **Use when**: You want to use only some columns of X
   - **What's NOT sampled**: W, M, Lambda, V (not needed)
   - **What IS sampled**: 
     - **Y layer**: `tau2_y` (MLE or MCMC), `g_y` (MLE or MCMC), `theta_y` (MLE or MCMC)
     - **Q layer (middle latent)**: `Q` (ESS), `theta_q` (MCMC only)
     - **Q layer fixed**: `g_q = 0.0`, `tau2_q = 1.0`
     - **R layer (inner latent)**: `R` (ESS), `theta_r` (MCMC only)
     - **R layer fixed**: `g_r = 0.0`, `tau2_r = 1.0`
   - **Config**: `create_config_L3_No_W_Selective(D, column_indices=None, **kwargs)`
   - **Usage**: `run_multichain_analysis(..., layer=3, variant='No_W_Selective', D=3, column_indices=[0,1,2], ...)`
   - **Model**: Y | Q, θ_y, g_y, τ² ~ GP(0, τ²(C_y + g_y*I)); Q | R, θ_q ~ GP(0, C_q); R | X[:, column_indices], θ_r ~ GP(0, C_r)

### All Layer 3 Variants Support:
- **Kernel type selection**: All four kernel types supported
- **Individual MLE options**: Only for Y layer (`use_mle_tau2`, `use_mle_g_y`, `use_mle_theta_y`)
- **R and Q layers**: Always use MCMC for `theta_r` and `theta_q` (no MLE option)
- **Hierarchical lengthscale priors**: 
  - `gamma2_y = 3.9` (outer layer Y)
  - `gamma2_q = 3.9/3` (middle layer Q)
  - `gamma2_r = 3.9/6` (inner layer R)
- **Both D=1 and D>1 cases**: Handles scalar vs vector theta automatically
- **Multi-chain diagnostics and metrics**: Full support


In [ ]:
# Example: Using Layer 1 Variants

# Generate data
data = generate_case1_1d(n=100, seed=42)
p = data['X_train'].shape[1]

# Example 1: W_Known variant
print("="*70)
print("Example 1: W_Known Variant")
print("="*70)

# Create a fixed W matrix
W_fixed = np.random.randn(p, 2)
W_fixed, _ = np.linalg.qr(W_fixed)

config_w_known = create_config_L1_W_Known(
    W_fixed=W_fixed,
    n_iterations=100,
    burn_in=20,
    use_mle_tau2=True,
    use_mle_g=False,
    use_mle_theta=True,
    kernel_type='separable_squared_exponential'
)

print(f"Config: variant={config_w_known['variant']}, D={config_w_known['D']}")
print(f"  W_fixed shape: {W_fixed.shape}")
print(f"  MLE: tau2={config_w_known['use_mle_tau2']}, g={config_w_known.get('use_mle_g', False)}, theta={config_w_known.get('use_mle_theta', False)}")

# Example 2: No_W variant
print("\n" + "="*70)
print("Example 2: No_W Variant")
print("="*70)

config_no_w = create_config_L1_No_W(
    n_iterations=100,
    burn_in=20,
    use_mle_tau2=False,
    use_mle_g=True,
    use_mle_theta=False,
    kernel_type='separable_squared_exponential'
)

print(f"Config: variant={config_no_w['variant']}")
print(f"  MLE: tau2={config_no_w['use_mle_tau2']}, g={config_no_w.get('use_mle_g', False)}, theta={config_no_w.get('use_mle_theta', False)}")

# Example 3: No_W_Selective variant
print("\n" + "="*70)
print("Example 3: No_W_Selective Variant")
print("="*70)

config_selective = create_config_L1_No_W_Selective(
    D=3,
    column_indices=np.array([0, 1, 2]),  # Use first 3 columns
    n_iterations=100,
    burn_in=20,
    use_mle_tau2=True,
    use_mle_g=True,
    use_mle_theta=True,
    kernel_type='separable_squared_exponential'
)

print(f"Config: variant={config_selective['variant']}, D={config_selective['D']}")
print(f"  column_indices: {config_selective['column_indices']}")
print(f"  MLE: tau2={config_selective['use_mle_tau2']}, g={config_selective.get('use_mle_g', False)}, theta={config_selective.get('use_mle_theta', False)}")

print("\n" + "="*70)
print("✅ All variant configurations created successfully!")
print("="*70)


In [ ]:
# Example: Using Layer 2 Variants

# Generate data
data = generate_case1_1d(n=100, seed=42)
p = data['X_train'].shape[1]

# Example 1: W_Known variant
print("="*70)
print("Example 1: W_Known Variant (Layer 2)")
print("="*70)

# Create a fixed W matrix
W_fixed = np.random.randn(p, 2)
W_fixed, _ = np.linalg.qr(W_fixed)

config_w_known = create_config_L2_W_Known(
    W_fixed=W_fixed,
    n_iterations=100,
    burn_in=20,
    use_mle_tau2=True,
    use_mle_g_y=False,
    use_mle_theta_y=True,
    kernel_type='separable_squared_exponential'
)

print(f"Config: variant={config_w_known['variant']}, D={config_w_known['D']}")
print(f"  W_fixed shape: {W_fixed.shape}")
print(f"  MLE: tau2_y={config_w_known['use_mle_tau2']}, g_y={config_w_known.get('use_mle_g_y', False)}, theta_y={config_w_known.get('use_mle_theta_y', False)}")

# Example 2: No_W variant
print("\n" + "="*70)
print("Example 2: No_W Variant (Layer 2)")
print("="*70)

config_no_w = create_config_L2_No_W(
    n_iterations=100,
    burn_in=20,
    use_mle_tau2=False,
    use_mle_g_y=True,
    use_mle_theta_y=False,
    kernel_type='separable_squared_exponential'
)

print(f"Config: variant={config_no_w['variant']}")
print(f"  MLE: tau2_y={config_no_w['use_mle_tau2']}, g_y={config_no_w.get('use_mle_g_y', False)}, theta_y={config_no_w.get('use_mle_theta_y', False)}")

# Example 3: No_W_Selective variant
print("\n" + "="*70)
print("Example 3: No_W_Selective Variant (Layer 2)")
print("="*70)

config_selective = create_config_L2_No_W_Selective(
    D=3,
    column_indices=np.array([0, 1, 2]),  # Use first 3 columns
    n_iterations=100,
    burn_in=20,
    use_mle_tau2=True,
    use_mle_g_y=True,
    use_mle_theta_y=True,
    kernel_type='separable_squared_exponential'
)

print(f"Config: variant={config_selective['variant']}, D={config_selective['D']}")
print(f"  column_indices: {config_selective['column_indices']}")
print(f"  MLE: tau2_y={config_selective['use_mle_tau2']}, g_y={config_selective.get('use_mle_g_y', False)}, theta_y={config_selective.get('use_mle_theta_y', False)}")

print("\n" + "="*70)
print("✅ All Layer 2 variant configurations created successfully!")
print("="*70)


In [ ]:
# Example: Running Multi-Chain Analysis with Layer 2 Variants

# Generate data
data = generate_case1_1d(n=100, seed=42)

# Example: W_Known variant (Layer 2)
print("="*70)
print("Running Multi-Chain Analysis: W_Known Variant (Layer 2)")
print("="*70)

# Create fixed W
W_fixed = np.random.randn(data['X_train'].shape[1], 2)
W_fixed, _ = np.linalg.qr(W_fixed)

# Run analysis with variant
results_w_known_L2 = run_multichain_analysis(
    Y_train=data['y_train'],
    X_train=data['X_train'],
    Y_test=data['y_test'],
    X_test=data['X_test'],
    layer=2,  # Layer 2
    variant='W_Known',  # Specify variant
    W_fixed=W_fixed,   # Required for W_Known
    n_chains=2,
    n_iterations=10,  # Short for demo
    burn_in=2,
    thin=1,
    use_mle_tau2=True,
    use_mle_g_y=False,
    use_mle_theta_y=True,
    kernel_type='separable_squared_exponential',
    verbose=True
)

print(f"\n✅ W_Known variant (Layer 2) complete!")
print(f"   Chains: {len(results_w_known_L2['chains_samples'])}")
print(f"   RMSPE: {results_w_known_L2['metrics_summary']['RMSPE']['mean']:.4f}")
print(f"   CP: {results_w_known_L2['metrics_summary']['CP']['mean']:.4f}")
print(f"   ALCI: {results_w_known_L2['metrics_summary']['ALCI']['mean']:.4f}")
print(f"   Iteration metrics length: {len(results_w_known_L2['chains_metrics'][0]['RMSPE_samples'])}")


In [ ]:
# Example: Using Layer 3 Variants

# Generate data
data = generate_case1_1d(n=100, seed=42)
p = data['X_train'].shape[1]

# Example 1: W_Known variant
print("="*70)
print("Example 1: W_Known Variant (Layer 3)")
print("="*70)

# Create a fixed W matrix
W_fixed = np.random.randn(p, 2)
W_fixed, _ = np.linalg.qr(W_fixed)

config_w_known = create_config_L3_W_Known(
    W_fixed=W_fixed,
    n_iterations=100,
    burn_in=20,
    use_mle_tau2=True,
    use_mle_g_y=False,
    use_mle_theta_y=True,
    kernel_type='separable_squared_exponential'
)

print(f"Config: variant={config_w_known['variant']}, D={config_w_known['D']}")
print(f"  W_fixed shape: {W_fixed.shape}")
print(f"  MLE: tau2_y={config_w_known['use_mle_tau2']}, g_y={config_w_known.get('use_mle_g_y', False)}, theta_y={config_w_known.get('use_mle_theta_y', False)}")

# Example 2: No_W variant
print("\n" + "="*70)
print("Example 2: No_W Variant (Layer 3)")
print("="*70)

config_no_w = create_config_L3_No_W(
    n_iterations=100,
    burn_in=20,
    use_mle_tau2=False,
    use_mle_g_y=True,
    use_mle_theta_y=False,
    kernel_type='separable_squared_exponential'
)

print(f"Config: variant={config_no_w['variant']}")
print(f"  MLE: tau2_y={config_no_w['use_mle_tau2']}, g_y={config_no_w.get('use_mle_g_y', False)}, theta_y={config_no_w.get('use_mle_theta_y', False)}")

# Example 3: No_W_Selective variant
print("\n" + "="*70)
print("Example 3: No_W_Selective Variant (Layer 3)")
print("="*70)

config_selective = create_config_L3_No_W_Selective(
    D=3,
    column_indices=np.array([0, 1, 2]),  # Use first 3 columns
    n_iterations=100,
    burn_in=20,
    use_mle_tau2=True,
    use_mle_g_y=True,
    use_mle_theta_y=True,
    kernel_type='separable_squared_exponential'
)

print(f"Config: variant={config_selective['variant']}, D={config_selective['D']}")
print(f"  column_indices: {config_selective['column_indices']}")
print(f"  MLE: tau2_y={config_selective['use_mle_tau2']}, g_y={config_selective.get('use_mle_g_y', False)}, theta_y={config_selective.get('use_mle_theta_y', False)}")

print("\n" + "="*70)
print("✅ All Layer 3 variant configurations created successfully!")
print("="*70)


In [ ]:
# Example: Running Multi-Chain Analysis with Layer 3 Variants

# Generate data
data = generate_case1_1d(n=100, seed=42)

# Example: W_Known variant (Layer 3)
print("="*70)
print("Running Multi-Chain Analysis: W_Known Variant (Layer 3)")
print("="*70)

# Create fixed W
W_fixed = np.random.randn(data['X_train'].shape[1], 2)
W_fixed, _ = np.linalg.qr(W_fixed)

# Run analysis with variant
results_w_known_L3 = run_multichain_analysis(
    Y_train=data['y_train'],
    X_train=data['X_train'],
    Y_test=data['y_test'],
    X_test=data['X_test'],
    layer=3,  # Layer 3
    variant='W_Known',  # Specify variant
    W_fixed=W_fixed,   # Required for W_Known
    n_chains=2,
    n_iterations=10,  # Short for demo
    burn_in=2,
    thin=1,
    use_mle_tau2=True,
    use_mle_g_y=False,
    use_mle_theta_y=True,
    kernel_type='separable_squared_exponential',
    verbose=True
)

print(f"\n✅ W_Known variant (Layer 3) complete!")
print(f"   Chains: {len(results_w_known_L3['chains_samples'])}")
print(f"   RMSPE: {results_w_known_L3['metrics_summary']['RMSPE']['mean']:.4f}")
print(f"   CP: {results_w_known_L3['metrics_summary']['CP']['mean']:.4f}")
print(f"   ALCI: {results_w_known_L3['metrics_summary']['ALCI']['mean']:.4f}")
print(f"   Iteration metrics length: {len(results_w_known_L3['chains_metrics'][0]['RMSPE_samples'])}")


In [ ]:
# Example: Running Multi-Chain Analysis with Layer 1 Variants

# Generate data
data = generate_case1_1d(n=100, seed=42)

# Example: W_Known variant
print("="*70)
print("Running Multi-Chain Analysis: W_Known Variant")
print("="*70)

# Create fixed W
W_fixed = np.random.randn(data['X_train'].shape[1], 2)
W_fixed, _ = np.linalg.qr(W_fixed)

# Run analysis with variant
results_w_known = run_multichain_analysis(
    Y_train=data['y_train'],
    X_train=data['X_train'],
    Y_test=data['y_test'],
    X_test=data['X_test'],
    layer=1,
    variant='W_Known',  # Specify variant
    W_fixed=W_fixed,   # Required for W_Known
    n_chains=2,
    n_iterations=10,  # Short for demo
    burn_in=2,
    thin=1,
    use_mle_tau2=True,
    use_mle_g=False,
    use_mle_theta=True,
    kernel_type='separable_squared_exponential',
    verbose=True
)

print(f"\n✅ W_Known variant complete!")
print(f"   Chains: {len(results_w_known['chains_samples'])}")
print(f"   RMSPE: {results_w_known['metrics_summary']['RMSPE']['mean']:.4f}")
print(f"   CP: {results_w_known['metrics_summary']['CP']['mean']:.4f}")
print(f"   ALCI: {results_w_known['metrics_summary']['ALCI']['mean']:.4f}")
print(f"   Iteration metrics length: {len(results_w_known['chains_metrics'][0]['RMSPE_samples'])}")


## Summary

This notebook demonstrated:

### ✅ Configuration Methods
1. **Preset functions with automatic initialization**: 
   - `create_config_D1_L1(p, seed, **kwargs)`
   - `create_config_D2_L2(p, seed, **kwargs)`
   - `create_config_D3_L3(p, seed, **kwargs)`
   - etc.
2. **Helper wrapper**: `get_config_for(D, layer, **overrides)`
3. **Manual config**: Specify all parameters directly
4. **Variant configs**: `create_config_L1/L2/L3_W_Known/No_W/No_W_Selective(...)`

### ✅ Key Features
- **D=1**: Scalar theta, simple 1D case
- **D>1**: Vector theta, separable kernels, TensorFlow gradients recommended
- **Layer 1 (Full)**: tau2_y, g_y, theta_D_y, W, M, V, Lambda
- **Layer 2 (Full)**: tau2_y, g_y, g_q, theta_y, theta_q, Q, W, M, V, Lambda
- **Layer 3 (Full)**: tau2_y, g_y, g_q, g_r, theta_y, theta_q, theta_r, R, Q, W, M, V, Lambda

### ✅ Layer Variants (Simplified Models)
- **Layer 1 Variants**: Skip W, M, Lambda, V; sample tau2_y, g_y, theta_D_y
- **Layer 2 Variants**: Skip W, M, Lambda, V; sample Q (ESS), theta_q (MCMC), tau2_y, g_y, theta_y (MLE/MCMC)
  - Q layer: g_q=0.0 (fixed), tau2_q=1.0 (fixed), theta_q (MCMC only)
- **Layer 3 Variants**: Skip W, M, Lambda, V; sample R (ESS), Q (ESS), theta_r (MCMC), theta_q (MCMC), tau2_y, g_y, theta_y (MLE/MCMC)
  - R layer: g_r=0.0 (fixed), tau2_r=1.0 (fixed), theta_r (MCMC only)
  - Q layer: g_q=0.0 (fixed), tau2_q=1.0 (fixed), theta_q (MCMC only)

### ✅ Automatic Initialization (NEW!)
- **Provide `p` parameter**: M, Lambda, V, W, and priors are automatically initialized
- **SVD-based procedure**: Uses `F = np.random.randn(p, D)`, SVD, then computes priors
- **For D=1**: Uses `rmf_matrixN` and `rmf_matrix` from `parameter_sampler_D1`
- **For D>1**: Uses `rmf_matrixN` and `rmf_matrix` from `parameter_sampler_Dgeneral`
- **prior_Lambda**: uses `Gamma(5/2, 10/3)`

### ✅ Available Presets
- **D=1**: L1, L2, L3 (all accept `p` and `seed`)
- **D=2**: L1, L2, L3 (all accept `p` and `seed`)
- **D=3**: L1, L2, L3 (all accept `p` and `seed`)
- **D=5**: L1, L2, L3 (all accept `p` and `seed`)
- **Layer Variants**: L1/L2/L3 with W_Known/No_W/No_W_Selective

### ✅ MLE Options
- **Layer 1**: Individual flags for `use_mle_tau2`, `use_mle_g`, `use_mle_theta` (default: False = MCMC)
- **Layer 2**: Individual flags for `use_mle_tau2`, `use_mle_g_y`, `use_mle_theta_y` (Q layer always MCMC)
- **Layer 3**: Individual flags for `use_mle_tau2`, `use_mle_g_y`, `use_mle_theta_y` (R and Q layers always MCMC)

### ✅ Hierarchical Lengthscale Priors (Multi-Layer)
- **Layer 2**: `gamma2_y = 3.9`, `gamma2_q = 3.9/3`
- **Layer 3**: `gamma2_y = 3.9`, `gamma2_q = 3.9/3`, `gamma2_r = 3.9/6`

### 📊 Diagnostic Plots
Check output directories for:
- Trace plots
- Density plots
- Convergence diagnostics
- Actual vs. Predicted
- Performance metrics


All configurations provide defaults that can be easily overridden!
**New**: Automatic initialization ensures consistent starting values when `p` is provided!
**New**: Variant models allow simplified analysis when W/M/Lambda/V are not of interest!
